# 04 · RLHF con PPO
*Optimizar contra el modelo de recompensa sin alejarse del punto de partida*

Notebook del episodio 4 de **Alineación a bajo nivel** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/ai-alignment/rlhf-con-ppo)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** muestrear → recompensar → penalizar KL con la referencia → ventaja → objetivo recortado de PPO

**Cómo correrlo:** con el entorno de `~/Movies/serie-alineacion/lab` (`cd lab && uv run --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "alineacion"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*Alineación a bajo nivel. Episodio cuatro: RLHF con PPO. Es decir, reinforcement learning from human feedback, con proximal policy optimization. Ya tenemos un modelo de SFT, supervised fine-tuning, que contesta y un modelo de recompensa que puntúa respuestas. Hoy los juntamos: vamos a ajustar la política para que saque más puntaje, con el algoritmo que usó InstructGPT. Y vamos a ver por qué el ingrediente más importante no es la recompensa, sino el freno.*

### Por qué hace falta aprendizaje por refuerzo

- La recompensa llega por respuesta entera, no por token
- No hay respuesta correcta para imitar: hay que probar
- Generar es muestrear: no se puede derivar a través del muestreo
- El RM es un proxy: si lo exprimís, se rompe

¿Por qué no seguir con entropía cruzada? Porque el modelo de recompensa puntúa la respuesta entera, no cada token. No hay una respuesta correcta para copiar: la política tiene que generar, recibir un puntaje y aprender de eso. Y generar es muestrear tokens, una operación discreta: el gradiente no pasa a través de un sorteo. Eso es exactamente el problema que resuelve el aprendizaje por refuerzo. Pero hay una trampa que vimos al final del episodio tres: el modelo de recompensa es un proxy. Si lo optimizás sin límite, encontrás sus agujeros.

## Parte 1 · El objetivo

Antes del algoritmo, la cuenta que queremos maximizar. Tiene dos términos, y el segundo es el importante.

### Recompensa menos distancia

$$
\max_{\pi}\; \mathbb{E}_{x,\; y \sim \pi}\big[r(x, y)\big] \;-\; \beta\, \mathrm{KL}\big(\pi(\cdot\mid x)\,\|\,\pi_{\text{ref}}(\cdot\mid x)\big)
$$
<small>$\pi_{\text{ref}}$: el modelo de SFT, congelado</small>

$$
\mathrm{KL}(\pi\|\pi_{\text{ref}}) = \mathbb{E}_{y\sim\pi}\Big[\log \frac{\pi(y\mid x)}{\pi_{\text{ref}}(y\mid x)}\Big]
$$
<small>cuánto se alejó la distribución</small>

$$
\log \pi(y\mid x) = \textstyle\sum_{t} \log \pi(y_t \mid x, y_{<t})
$$
<small>la de la respuesta es la suma de los tokens</small>

El objetivo de RLHF es este. Maximizar la recompensa esperada de las respuestas que genera la política, pi, menos beta por la divergencia KL, de Kullback-Leibler, entre la política y el modelo de referencia, pi sub ref, que es el de SFT congelado. Beta es la perilla: cuánto te importa no alejarte. La KL es el valor esperado del logaritmo del cociente de probabilidades, sobre respuestas que genera la política. Y como la probabilidad de una respuesta es el producto de las de sus tokens, su logaritmo es una suma. Eso nos va a permitir calcular la KL token por token. En el dibujo, una respuesta real: el log cociente de cada token, uno es negativo, y la suma da casi diez.

### Las piezas de RLHF

Este es el mapa del episodio. Tomamos prompts y la política genera respuestas. El modelo de recompensa las puntúa, el de referencia calcula sus log probabilidades, y combinamos las dos cosas en una recompensa penalizada. Un cuarto modelo, el crítico, estima cuánto puntaje se espera desde cada token, y con eso calculamos ventajas con GAE, generalized advantage estimation: qué tokens fueron mejores de lo esperado. Y PPO da un paso, con cuidado de no moverse demasiado. Cuatro modelos en memoria: política, referencia, recompensa y crítico. En InstructGPT, la recompensa y el crítico eran de seis mil millones de parámetros cada uno.

prompts → política π genera → RM puntúa → π_ref: log-probs → recompensa − β·KL → crítico V → ventajas (GAE) → paso de PPO

## Parte 2 · La KL, token por token

El freno se calcula con las mismas log-probabilidades que ya sabés sacar de los logits.

### Log-probs y KL por token

Para este episodio la política es un modelo de juguete al que le movimos un poco los pesos respecto de la referencia. Calculamos log softmax sobre los logits: la distribución completa en cada posición. Con gather nos quedamos con el log-prob del token que efectivamente salió. La diferencia entre la política y la referencia, en cada token, es una estimación de la KL con una sola muestra. Sumada y promediada sobre muchas respuestas, converge a la KL de verdad. En la primera secuencia hay un token negativo: un token suelto puede dar menos que cero, aunque la KL nunca lo es. Pasa en el trece por ciento de los tokens. Pero el promedio da ocho coma tres nueve, contra ocho coma tres seis de la KL exacta.

**Cada línea, su término:**

- `log_softmax(-1)` → $\log \pi(\cdot \mid x, y_{<t})$ — toda la distribución en cada posición
- `gather` → $\log \pi(y_t \mid x, y_{<t})$ — nos quedamos con el token que salió
- `kl_tok =` → $k_t = \log \pi(y_t) - \log \pi_{\text{ref}}(y_t)$ — estimador de la KL con una muestra
- `kl_tok.sum(-1).mean()` → $\mathbb{E}_{y\sim\pi}\big[\textstyle\sum_t k_t\big] = \mathrm{KL}$ — en promedio, da la KL exacta

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep04_kl.py
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import LlamaConfig, LlamaForCausalLM
torch.manual_seed(0)
V, P, N = 16, 2, 6                                    # vocabulario, tokens de prompt, tokens de respuesta
w_oculto = torch.randn(V)
def rm(y):                                            # el "modelo de recompensa": el gusto oculto del ep. 3
    return w_oculto[y].sum(-1) * 0.7
cfg = LlamaConfig(vocab_size=V, hidden_size=64, intermediate_size=128, num_hidden_layers=2,
                  num_attention_heads=4, num_key_value_heads=4, max_position_embeddings=32)
ref = LlamaForCausalLM(cfg).eval()                    # π_ref: el modelo de partida (congelado)
for p_ in ref.parameters():
    p_.requires_grad_(False)

@torch.no_grad()
def muestrear(model, x, n=N):                         # muestreo token a token, temperatura 1
    for _ in range(n):
        p = model(input_ids=x).logits[:, -1].softmax(-1)
        x = torch.cat([x, torch.multinomial(p, 1)], 1)
    return x

def prompts(n):
    return torch.randint(2, V, (n, P))
pol = copy.deepcopy(ref)                              # una política que ya se movió un poco
with torch.no_grad():
    pol.lm_head.weight += 0.3 * torch.randn_like(pol.lm_head.weight)

In [3]:
def logprobs(model, ids):              # log π(y_t | x, y_<t) por token
    lp = model(input_ids=ids).logits[:, :-1].log_softmax(-1)
    lp = lp.gather(-1, ids[:, 1:, None]).squeeze(-1)  # el que salió
    return lp[:, P - 1:]                   # solo la respuesta

y = muestrear(pol, prompts(2000))                    # respuestas de la política
with torch.no_grad():
    kl_tok = logprobs(pol, y) - logprobs(ref, y)     # log π − log π_ref
print("seq 0, por token:", kl_tok[0].numpy().round(2))
print(f"tokens negativos: {(kl_tok < 0).float().mean():.0%}")
print(f"estimador (promedio): {kl_tok.sum(-1).mean():.3f}")

seq 0, por token: [ 1.14  2.18  1.89  2.66 -0.15  2.27]
tokens negativos: 13%
estimador (promedio): 8.389


In [4]:
# Verificación extra (corrió junto con el video)
with torch.no_grad():                                 # KL exacta, sumando sobre el vocabulario
    lq = pol(input_ids=y).logits[:, P - 1:-1].log_softmax(-1)
    lr = ref(input_ids=y).logits[:, P - 1:-1].log_softmax(-1)
    exacta = (lq.exp() * (lq - lr)).sum(-1).sum(-1).mean()
print(f"KL exacta (promedio): {exacta:.3f}")

KL exacta (promedio): 8.363


### La recompensa con el freno adentro

Ahora metemos el freno en la recompensa. Cada token recibe menos beta por su estimación de KL: si la política le dio a ese token mucha más probabilidad que la referencia, paga. Y el puntaje del modelo de recompensa se suma solo en el último token, porque recién ahí la respuesta está completa. La suma de todo es exactamente el objetivo de la escena anterior, para una respuesta. Con beta cero coma uno, los tres primeros tokens reciben recompensas chiquitas, positivas o negativas, y el último uno coma diecinueve. Total: uno coma ciento cincuenta y siete, el uno coma dos del RM menos cero coma cero cuarenta y tres de freno.

**Cada línea, su término:**

- `r = -beta * kl_tok` → $r_t = -\beta\, k_t$ — cada token paga por alejarse
- `r[:, -1] += score` → $r_T \mathrel{+}= r_{\text{RM}}(x, y)$ — el RM puntúa al final
- `print(f"total` → $\textstyle\sum_t r_t = r_{\text{RM}} - \beta \sum_t k_t$ — la cuenta del objetivo, por respuesta

In [5]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep04_recompensa.py
import torch
torch.manual_seed(0)
score = torch.tensor([1.2])                           # lo que dijo el RM de la respuesta entera
kl_tok = torch.tensor([[0.05, -0.02, 0.30, 0.10]])    # log π − log π_ref, token a token

In [6]:
def recompensa(score, kl_tok, beta=0.1):
    r = -beta * kl_tok                  # penalización en cada token
    r[:, -1] += score                   # el RM puntúa al final
    return r

r = recompensa(score, kl_tok)
print("r por token:", r.numpy().round(3))
print(f"total: {r.sum():.3f} = {score.item():.1f} − 0.1 · {kl_tok.sum():.2f}")

r por token: [[-0.005  0.002 -0.03   1.19 ]]
total: 1.157 = 1.2 − 0.1 · 0.43


> **Ouyang et al., 2022 — Training language models to follow instructions with human feedback** · [arXiv:2203.02155](https://arxiv.org/abs/2203.02155)
>
> **β = 0,02** · InstructGPT agrega una penalización KL por token respecto del modelo de SFT para mitigar la sobreoptimización del RM, con β = 0,02; ratio de recorte de PPO 0,2.

Esto es lo que hizo InstructGPT. Una penalización de KL por token respecto del modelo de SFT, con la razón explícita de mitigar la sobreoptimización del modelo de recompensa. Usaron beta igual a cero coma cero dos, y un recorte de PPO de cero coma dos, que vamos a ver en un momento.

## Parte 3 · Repartir el crédito

La recompensa llega al final: ¿qué token tuvo la culpa? Para eso está el crítico.

### Ventajas con GAE

$$
\delta_t = r_t + \gamma\, V(s_{t+1}) - V(s_t)
$$
<small>sorpresa de un paso: lo que pasó menos lo esperado</small>

$$
\hat{A}_t = \delta_t + \gamma\lambda\, \hat{A}_{t+1} = \textstyle\sum_{l \ge 0} (\gamma\lambda)^l\, \delta_{t+l}
$$
<small>suma de sorpresas futuras, descontadas</small>

$$
R_t = \hat{A}_t + V(s_t)
$$
<small>el blanco para entrenar al crítico</small>

El crítico, V, es un value head: un número, como el reward head del episodio tres, pero en cada posición: estima cuánto puntaje total se espera desde ese punto de la respuesta. La sorpresa de un paso, delta, es la recompensa que llegó más lo que se espera desde el siguiente estado, menos lo que se esperaba antes. La ventaja estimada, con GAE, acumula las sorpresas futuras con un descuento gama por lambda. En el dibujo, el primer token: cuatro sorpresas, cada una más descontada, suman cero coma siete. Lambda negocia entre ruido y sesgo. InstructGPT usa gama igual a uno, sin descuento. Y la ventaja más el valor da el retorno, que es el blanco con el que se entrena al crítico.

### GAE en ocho líneas

En código. El cálculo va de atrás para adelante, porque cada ventaja depende de la siguiente. La sorpresa de un paso. La recurrencia: la ventaja de este token es su sorpresa más gama lambda por la ventaja del siguiente. Y devolvemos ventajas y retornos. Usamos las recompensas de la escena anterior y un crítico que esperaba poco al principio. El primer token tiene ventaja cero coma siete: el crítico esperaba cero coma cuatro y la respuesta terminó valiendo bastante más. El último, cero coma diecinueve: ahí el crítico ya esperaba casi todo. La recompensa del final se repartió hacia atrás.

In [7]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep04_gae.py
import torch
r = torch.tensor([[-0.005, 0.002, -0.030, 1.190]])    # recompensas por token (escena anterior)
v = torch.tensor([[0.40, 0.50, 0.90, 1.00]])          # lo que el crítico esperaba en cada paso

In [8]:
def gae(r, v, gamma=1.0, lam=0.95):
    A, ultimo, T = torch.zeros_like(r), 0.0, r.shape[1]
    for t in reversed(range(T)):                     # de atrás para adelante
        v_sig = v[:, t + 1] if t + 1 < T else 0.0
        delta = r[:, t] + gamma * v_sig - v[:, t]    # δ_t: sorpresa de un paso
        ultimo = delta + gamma * lam * ultimo        # A_t = δ_t + γλ A_{t+1}
        A[:, t] = ultimo
    return A, A + v                                  # ventajas y retornos

A, R = gae(r, v)
print("ventajas A:", A.numpy().round(3))
print("retornos R:", R.numpy().round(3))

ventajas A: [[0.703 0.64  0.251 0.19 ]]
retornos R: [[1.103 1.14  1.15  1.19 ]]


## Parte 4 · El paso recortado de PPO

Con las ventajas ya sabemos qué tokens reforzar. PPO decide cuánto, sin dar pasos demasiado largos.

### Gradiente de política y recorte

$$
\nabla_\theta J = \mathbb{E}\big[\nabla_\theta \log \pi_\theta(y_t \mid s_t)\, \hat{A}_t\big]
$$
<small>subir la log-prob de lo que salió mejor de lo esperado</small>

$$
\rho_t = \frac{\pi_\theta(y_t \mid s_t)}{\pi_{\text{old}}(y_t \mid s_t)}
$$
<small>cuánto cambió la política desde que generó</small>

$$
\mathcal{L}^{\text{CLIP}} = -\,\mathbb{E}\big[\min\big(\rho_t \hat{A}_t,\; \mathrm{clip}(\rho_t, 1-\epsilon, 1+\epsilon)\, \hat{A}_t\big)\big]
$$
<small>$\epsilon = 0{,}2$: el ratio vive en $[0{,}8;\,1{,}2]$</small>

La base es el gradiente de política. Multiplicás el gradiente de la log probabilidad de cada token por su ventaja: si salió mejor de lo esperado, lo hacés más probable. Así se esquiva el muestreo: no derivás a través del sorteo, derivás la probabilidad de lo que salió. PPO reutiliza cada lote para varios pasos, y para eso mide cuánto cambió la política desde que generó: el ratio rho, nueva sobre vieja. Y la pérdida toma el mínimo entre el término normal y uno con el ratio recortado entre cero coma ocho y uno coma dos. Una vez que un token ya cambió lo suficiente en la dirección buena, deja de recibir empuje.

### El recorte, caso por caso

Veámoslo con seis tokens. El ratio es la exponencial de la diferencia de log-probs. La versión recortada. El mínimo de las dos, con el signo cambiado. Y miramos qué tokens reciben gradiente. Con ventaja positiva y ratio uno coma cinco, no empuja: ese token ya subió de más. Con ventaja negativa y ratio cero coma cinco, tampoco: ya bajó bastante. Pero con ventaja negativa y ratio uno coma cinco, sí empuja: la política subió un token que era malo, y el mínimo elige el término sin recortar para corregirlo. El recorte frena las mejoras, nunca los errores.

In [9]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep04_ppo.py
import torch
lp_old = torch.zeros(6)                               # log π_old de seis tokens
lp = torch.log(torch.tensor([0.5, 1.0, 1.5, 0.5, 1.0, 1.5])).requires_grad_(True)
A = torch.tensor([1., 1., 1., -1., -1., -1.])         # ventaja de cada token

In [10]:
def perdida_ppo(lp, lp_old, A, eps=0.2):
    ratio = (lp - lp_old).exp()                       # π / π_old
    s1 = ratio * A
    s2 = ratio.clamp(1 - eps, 1 + eps) * A            # recortado a [0.8, 1.2]
    return -torch.min(s1, s2).mean(), ratio

loss, ratio = perdida_ppo(lp, lp_old, A)
loss.backward()
print("ratio:   ", ratio.detach().numpy().round(2))
print("ventaja: ", A.numpy())
print("¿empuja?:", (lp.grad != 0).numpy())

ratio:    [0.5 1.  1.5 0.5 1.  1.5]
ventaja:  [ 1.  1.  1. -1. -1. -1.]
¿empuja?: [ True  True False False  True  True]


> **Schulman et al., 2017 — Proximal Policy Optimization Algorithms** · [arXiv:1707.06347](https://arxiv.org/abs/1707.06347)
>
> **0,82 vs. −0,39** · En su benchmark de control continuo (7 entornos, 21 corridas), recortar con $\epsilon = 0{,}2$ dio un puntaje normalizado promedio de 0,82; sin recorte ni penalización, −0,39.

El paper de PPO comparó variantes en tareas de control continuo, con puntajes normalizados donde cero es la política al azar. Recortando con épsilon cero coma dos, el promedio fue cero coma ochenta y dos, el mejor de su tabla. Sin recorte ni penalización, menos cero coma treinta y nueve: peor que no entrenar. Reusar datos sin freno rompe la política.

## Parte 5 · El loop completo

Ya tenemos todas las piezas. Juntémoslas y miremos qué hace beta.

### RLHF con PPO, de punta a punta

Este es el loop entero, con beta cero coma uno. La política genera respuestas de seis tokens. El modelo de recompensa las puntúa; acá, un gusto oculto como el del episodio tres. Restamos beta por la KL con la referencia. El crítico da las ventajas, que además normalizamos, una práctica habitual. Cuatro pasadas sobre el mismo lote. Y la pérdida suma la de PPO con el error cuadrático del crítico. El ratio de la primera pasada da exactamente uno: política vieja y nueva son la misma. Ese es el chequeo de cordura. La recompensa sube de menos cero coma tres cuatro a cuatro coma nueve uno, cerca del máximo de cinco coma dos. Y la KL, de cero a once.

In [11]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep04_loop.py
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import LlamaConfig, LlamaForCausalLM
torch.manual_seed(0)
V, P, N = 16, 2, 6                                    # vocabulario, tokens de prompt, tokens de respuesta
w_oculto = torch.randn(V)
def rm(y):                                            # el "modelo de recompensa": el gusto oculto del ep. 3
    return w_oculto[y].sum(-1) * 0.7
cfg = LlamaConfig(vocab_size=V, hidden_size=64, intermediate_size=128, num_hidden_layers=2,
                  num_attention_heads=4, num_key_value_heads=4, max_position_embeddings=32)
ref = LlamaForCausalLM(cfg).eval()                    # π_ref: el modelo de partida (congelado)
for p_ in ref.parameters():
    p_.requires_grad_(False)

@torch.no_grad()
def muestrear(model, x, n=N):                         # muestreo token a token, temperatura 1
    for _ in range(n):
        p = model(input_ids=x).logits[:, -1].softmax(-1)
        x = torch.cat([x, torch.multinomial(p, 1)], 1)
    return x

def logprobs(model, ids):                             # log π(y_t | x, y_<t) de cada token de la respuesta
    lp = model(input_ids=ids).logits[:, :-1].log_softmax(-1)
    lp = lp.gather(-1, ids[:, 1:, None]).squeeze(-1)  # el log-prob del token que salió
    return lp[:, P - 1:]                              # solo las N posiciones de la respuesta

def prompts(n):
    return torch.randint(2, V, (n, P))

def recompensa(score, kl_tok, beta=0.1):
    r = -beta * kl_tok
    r[:, -1] += score
    return r

def gae(r, v, gamma=1.0, lam=0.95):
    A, ultimo, T = torch.zeros_like(r), 0.0, r.shape[1]
    for t in reversed(range(T)):
        v_sig = v[:, t + 1] if t + 1 < T else 0.0
        delta = r[:, t] + gamma * v_sig - v[:, t]
        ultimo = delta + gamma * lam * ultimo
        A[:, t] = ultimo
    return A, A + v

def perdida_ppo(lp, lp_old, A, eps=0.2):
    ratio = (lp - lp_old).exp()
    return -torch.min(ratio * A, ratio.clamp(1 - eps, 1 + eps) * A).mean()

class Critico(nn.Module):                             # V(s_t): una cabeza escalar por posición
    def __init__(self):
        super().__init__()
        self.cuerpo = LlamaForCausalLM(cfg).model
        self.cabeza = nn.Linear(cfg.hidden_size, 1)
    def forward(self, ids):
        h = self.cuerpo(input_ids=ids).last_hidden_state[:, P - 1:-1]
        return self.cabeza(h).squeeze(-1)
beta = 0.1
torch.manual_seed(0)
pol, critico = copy.deepcopy(ref).train(), Critico()
for p_ in pol.parameters():
    p_.requires_grad_(True)
opt = torch.optim.AdamW([*pol.parameters(), *critico.parameters()], lr=1e-3)

In [12]:
for it in range(61):
    y = muestrear(pol, prompts(64))                  # 1. la política genera
    with torch.no_grad():
        lp_old, lp_ref = logprobs(pol, y), logprobs(ref, y)
        score = rm(y[:, P:])                          # 2. el RM puntúa
        r = recompensa(score, lp_old - lp_ref, beta)  # 3. menos β·KL
        A, R = gae(r, critico(y))                     # 4. ventajas
        A = (A - A.mean()) / (A.std() + 1e-8)
    for epoca in range(4):                            # 5. varias pasadas
        lp = logprobs(pol, y)
        if it == 0 and epoca == 0:
            print("ratio inicial:", (lp - lp_old).exp().mean().item())
        loss = perdida_ppo(lp, lp_old, A) + 0.5 * F.mse_loss(critico(y), R)
        loss.backward(); opt.step(); opt.zero_grad()
    if it % 20 == 0:
        kl = (lp_old - lp_ref).sum(-1).mean()
        print(f"it {it:2d}  RM {score.mean():+.2f}  KL {kl:.2f}")

ratio inicial: 1.0


it  0  RM -0.34  KL 0.00


it 20  RM +4.88  KL 10.99


it 40  RM +4.94  KL 11.76


it 60  RM +4.91  KL 11.28


In [13]:
# Verificación extra (corrió junto con el video)
print(f"máximo posible del RM: {6 * 0.7 * w_oculto.max():+.2f}")

máximo posible del RM: +5.20


### La perilla β

Ahora corremos el mismo loop con cuatro valores de beta. De cero, sin freno, a uno. Para cada uno, medimos la recompensa media, la KL, y mostramos una respuesta de ejemplo. Con beta cero la recompensa es la máxima, cinco coma dos, y la respuesta es el token trece repetido seis veces: el preferido del modelo de recompensa. Es un colapso. Con cero coma uno, lo mismo, con algo menos de KL. Con cero coma tres, la respuesta vuelve a variar y la KL cae a cinco. Con beta uno, casi no se movió.

In [14]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep04_beta.py
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import LlamaConfig, LlamaForCausalLM
torch.manual_seed(0)
V, P, N = 16, 2, 6                                    # vocabulario, tokens de prompt, tokens de respuesta
w_oculto = torch.randn(V)
def rm(y):                                            # el "modelo de recompensa": el gusto oculto del ep. 3
    return w_oculto[y].sum(-1) * 0.7
cfg = LlamaConfig(vocab_size=V, hidden_size=64, intermediate_size=128, num_hidden_layers=2,
                  num_attention_heads=4, num_key_value_heads=4, max_position_embeddings=32)
ref = LlamaForCausalLM(cfg).eval()                    # π_ref: el modelo de partida (congelado)
for p_ in ref.parameters():
    p_.requires_grad_(False)

@torch.no_grad()
def muestrear(model, x, n=N):                         # muestreo token a token, temperatura 1
    for _ in range(n):
        p = model(input_ids=x).logits[:, -1].softmax(-1)
        x = torch.cat([x, torch.multinomial(p, 1)], 1)
    return x

def logprobs(model, ids):                             # log π(y_t | x, y_<t) de cada token de la respuesta
    lp = model(input_ids=ids).logits[:, :-1].log_softmax(-1)
    lp = lp.gather(-1, ids[:, 1:, None]).squeeze(-1)  # el log-prob del token que salió
    return lp[:, P - 1:]                              # solo las N posiciones de la respuesta

def prompts(n):
    return torch.randint(2, V, (n, P))

def recompensa(score, kl_tok, beta=0.1):
    r = -beta * kl_tok
    r[:, -1] += score
    return r

def gae(r, v, gamma=1.0, lam=0.95):
    A, ultimo, T = torch.zeros_like(r), 0.0, r.shape[1]
    for t in reversed(range(T)):
        v_sig = v[:, t + 1] if t + 1 < T else 0.0
        delta = r[:, t] + gamma * v_sig - v[:, t]
        ultimo = delta + gamma * lam * ultimo
        A[:, t] = ultimo
    return A, A + v

def perdida_ppo(lp, lp_old, A, eps=0.2):
    ratio = (lp - lp_old).exp()
    return -torch.min(ratio * A, ratio.clamp(1 - eps, 1 + eps) * A).mean()

class Critico(nn.Module):                             # V(s_t): una cabeza escalar por posición
    def __init__(self):
        super().__init__()
        self.cuerpo = LlamaForCausalLM(cfg).model
        self.cabeza = nn.Linear(cfg.hidden_size, 1)
    def forward(self, ids):
        h = self.cuerpo(input_ids=ids).last_hidden_state[:, P - 1:-1]
        return self.cabeza(h).squeeze(-1)

def rlhf(beta, iters=61):
    torch.manual_seed(0)
    pol, critico = copy.deepcopy(ref).train(), Critico()
    for p_ in pol.parameters():
        p_.requires_grad_(True)
    opt = torch.optim.AdamW([*pol.parameters(), *critico.parameters()], lr=1e-3)
    for it in range(iters):
        y = muestrear(pol, prompts(64))
        with torch.no_grad():
            lp_old, lp_ref = logprobs(pol, y), logprobs(ref, y)
            score = rm(y[:, P:])
            A, R = gae(recompensa(score, lp_old - lp_ref, beta), critico(y))
            A = (A - A.mean()) / (A.std() + 1e-8)
        for _ in range(4):
            loss = perdida_ppo(logprobs(pol, y), lp_old, A) + 0.5 * ((critico(y) - R) ** 2).mean()
            loss.backward(); opt.step(); opt.zero_grad()
    y = muestrear(pol, prompts(1000))
    with torch.no_grad():
        kl = (logprobs(pol, y) - logprobs(ref, y)).sum(-1).mean()
    return rm(y[:, P:]).mean().item(), kl.item(), y[0, P:].tolist()

In [15]:
for beta in [0.0, 0.1, 0.3, 1.0]:
    score, kl, ejemplo = rlhf(beta)
    print(f"β={beta:<4} RM {score:+.2f}  KL {kl:5.2f}  {ejemplo}")

β=0.0  RM +5.20  KL 17.22  [13, 13, 13, 13, 13, 13]


β=0.1  RM +4.93  KL 11.60  [13, 13, 13, 13, 13, 13]


β=0.3  RM +3.74  KL  5.23  [2, 13, 5, 4, 13, 4]


β=1.0  RM +1.63  KL  1.09  [2, 13, 8, 2, 0, 4]


In [16]:
# Verificación extra (corrió junto con el video)
print("token preferido por el RM:", int(w_oculto.argmax()))

token preferido por el RM: 13


### Cuánta recompensa compra cada unidad de KL

Puesto en un gráfico. Cada punto es un beta: a menos freno, más KL y más recompensa, pero con rendimientos decrecientes. Y la recompensa se pega contra el máximo del modelo de recompensa justo cuando la política colapsa. En un modelo real, ese máximo no es una respuesta perfecta: es un agujero del proxy. El paper de Anthropic sobre RLHF reporta una relación aproximadamente lineal entre la recompensa y la raíz cuadrada de la KL; en el episodio seis vemos qué pasa cuando se sigue empujando.

<small>Un punto por β (1; 0,3; 0,1; 0). Punteada: máximo del RM, 5,20. Salida real de ep04_beta.</small>

### Lo que cuesta RLHF con PPO

- Cuatro modelos en memoria: π, π_ref, RM y crítico
- Muchos hiperparámetros: β, ε, λ, épocas, tasa
- Sin β, colapsa sobre el agujero del RM
- ¿Hace falta el RM explícito? (episodio 5)

Antes de cerrar, el costo. Cuatro modelos en memoria a la vez. Una lista larga de hiperparámetros que interactúan: beta, épsilon, lambda, épocas, tasa de aprendizaje. Y cada iteración necesita generar, que es lo más lento. Sin el freno de la KL, la política encuentra el agujero del modelo de recompensa y se queda ahí. La pregunta natural es si todo este aparato hace falta. Resulta que el objetivo con KL tiene una solución en forma cerrada, y de ahí sale DPO, direct preference optimization, que entrena directo sobre las preferencias, sin modelo de recompensa ni crítico.

## Lo que te llevás

- Objetivo: $\mathbb{E}[r] - \beta\,\mathrm{KL}(\pi\|\pi_{\text{ref}})$
- KL por token: $\log\pi - \log\pi_{\text{ref}}$ del token muestreado
- Recompensa: $-\beta k_t$ en cada token, más el RM al final
- GAE: sorpresas $\delta_t$ acumuladas hacia atrás
- PPO: ratio recortado a $[1-\epsilon, 1+\epsilon]$
- Próximo: DPO y la constitución